## Task 3 part 2 - Modelling 1 (Baseline)

In [1]:
# load back in the data from Task3_01
import pandas as pd

DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f'{DATA_DIR}/task3_pert_FC_selected_50.pkl')
train_40 = pd.read_csv(f'{DATA_DIR}/task3_train_40.csv')['perturbation'].tolist()
test_10 = pd.read_csv(f'{DATA_DIR}/task3_test_10.csv')['perturbation'].tolist()

In [2]:
pert_FC_train = pert_FC_selected.loc[train_40].copy()
pert_FC_test = pert_FC_selected.loc[test_10].copy()

Now that we have the data sliced down to 50 perturbations and split into training and testing perturbations we will train our first model. This one is supposed to be "deliberately simplistic", as stated in the task description. 
Therefore we will call this the baseline model which for a test gene predicts its log2FC is just the average over all training perturbations in that condition. 

In [ ]:
conditions = ['Control', 'IFNγ', 'Co-culture']
# compute the baseline prediction for each condition (for each gene the mean over all perturbations in that condition is taken)
baseline_predictions  = {}
for cond in conditions:
    baseline_predictions[cond] = pert_FC_train.xs(cond, level = 'condition').mean(axis=0)

Now that we have computed the predictions for the test pertrubations as the average across all train perturbations in each conditions we will need to evaluate the performance of this very simplistic baseline model. For this we compute the correlation of the predicted log2FC vector and the actual log2FC vectors for each perturbation in each condition as well as the mean squared error as well as the standard deviations both across all conditions and for each condition separately. 

In [26]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

def evaluate_predictions_baseline(true_df, predictions_by_condition):
    # initialize records list
    records = []
    # iterate over testing data (two level index + FC vector)
    for (pert, cond), true_fc in true_df.iterrows():
        # predicted FC vectors from training are equal to average in each condition
        pred_fc = predictions_by_condition[cond]
        # compute pearson and spearman correlation
        pearson_r, pearson_p = pearsonr(true_fc, pred_fc)
        spearman_r, spearman_p = spearmanr(true_fc, pred_fc)
        # compute mse
        mse = np.mean((true_fc - pred_fc) ** 2)
        # append values to record list 
        records.append({
            "perturbation": pert,
            "condition": cond,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p,
            "mse": mse,
        })
    return pd.DataFrame(records)

In [30]:
# per (perturbation, condition) scores on the test perturbations
baseline_eval = evaluate_predictions_baseline(pert_FC_test, baseline_predictions)

metrics = ["pearson_r", "spearman_r", "mse"]

# per-condition mean and std of corrleation and mse values (n=10) for biological interpretation
per_condition_baseline = baseline_eval.groupby("condition")[metrics].agg(["mean", "std"])

# pooled mean and std across all test pairs (n=30)
overall_baseline = baseline_eval[metrics].agg(["mean", "std"])

baseline_eval

,perturbation,condition,pearson_r,pearson_p,spearman_r,spearman_p,mse
0,KCNN4,Control,0.828348,0.000000,0.457212,5.038036e-106,0.001508
1,KCNN4,IFNγ,0.844713,0.000000,0.399161,5.798557e-79,0.001100
2,KCNN4,Co-culture,0.862499,0.000000,0.339077,4.081474e-56,0.001128
3,TIMM50,Control,0.741626,0.000000,0.392917,2.335284e-76,0.002767
4,TIMM50,IFNγ,0.624917,0.000000,0.294308,4.385691e-42,0.003312
5,TIMM50,Co-culture,0.698907,0.000000,0.194893,6.285666e-19,0.003830
6,TXNDC17,Control,0.805104,0.000000,0.492349,3.787659e-125,0.004353
7,TXNDC17,IFNγ,0.620988,0.000000,0.446131,1.922074e-100,0.004324
8,TXNDC17,Co-culture,0.762693,0.000000,0.396490,7.660523e-78,0.004462
9,CORO1A,Control,0.806694,0.000000,0.375133,3.066797e-69,0.001195


In [31]:
overall_baseline

,pearson_r,spearman_r,mse
mean,0.688396,0.348603,0.002966
std,0.328608,0.119509,0.003637


In [29]:
per_condition_baseline

pearson_r           spearman_r                 mse          
                mean       std       mean       std      mean       std
condition                                                              
Co-culture  0.592829  0.498193   0.267087  0.107349  0.003967  0.005891
Control     0.754654  0.137720   0.398926  0.117838  0.002500  0.001594
IFNγ        0.717705  0.254562   0.379795  0.096609  0.002430  0.001927

## Discussion

In this notebook we "trained" and evaluated a deliberately simplistic baseline model: for a held-out test perturbation in a given condition, predict its log2FC fingerprint the average log2FC-vector across the 40 training perturbations in that same condition. No information about the specific test gene is used at all.

This resulted in an average Pearson correlation of r = 0.688, a Spearman correlation of  r = 0.349, and an MSE = 0.00297.

Pearson is noticeably higher than Spearman throughout. Pearson correlation is high for this model as many genes show a general/shared response that is highly similar for each perturbation, which makes them good to predict via the mean. Also they show exhibit large FC values on average since they point in the same direction (shared) which inflates Pearson correlation. However, many genes show a near zero fold change on average because they are regulated up or down in a perturbation specific manner or do not react to any perturbation which is why their average FC values only differ by nuances which does not influence Pearson correlation too much but Spearman correlation on the other hand uses ranks so it also detects minor differences. Naturally the exact ordering of these near zero FC genes is hard to predict using the average which is why the Spearman correlation is much lower. 

When evaluating the model on each of the conditions separately. o-culture is both the worst-performing (Pearson 0.593) and the most variable (std 0.498) condition, while control is the best and most consistent (Pearson 0.755, lowest std) followed closely by the IFN-$\gamma$ treated condition. This shows that there is a quite uniform response in both of these conditions wich expose the cells to either no or one single interferon.Under co-cultured conditions the response is much more specific as they face much more immune interactions so the different perturbations diverge more in their effect on the set of immune related genes from the mean which is why the model performs more poorly.

This baseline is a useful reference point precisely because it's not trying to use any gene-specific signal which is a step to be taken by any further models in order to outperform this model.